In [1]:
from xgboost import XGBRegressor as XGBR 
from sklearn.ensemble import RandomForestRegressor as RFR 
from sklearn.linear_model import LinearRegression as LinearR 
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold, cross_val_score as CVS, train_test_split as TTS 
from sklearn.metrics import mean_squared_error as MSE 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from time import time
import datetime
import os

p1: number of trees
- xgboost: num_round (default = 10), usually need to increase to 100
- sklearn: n_estimators (default = 100)

p2: silent (whether output training results in each iteration)
- xgboost: silent (default = False)
- sklearn: silent (default = True)


In [3]:
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
boston_local_data = np.loadtxt('data/housing.csv')
X = boston_local_data[:, 0:13]
y = boston_local_data[:, 13]

In [4]:
X.shape

(506, 13)

In [5]:
y.shape

(506,)

In [6]:
Xtrain, Xtest, Ytrain, Ytest = TTS(X, y, test_size=0.3, random_state=420)

In [8]:
# modelling
reg = XGBR(n_estimators=100).fit(Xtrain, Ytrain) # training
reg.predict(Xtest)

array([ 6.1930776, 21.161926 , 31.98226  , 14.511949 ,  9.299949 ,
       21.354044 , 16.132528 , 15.7151785, 15.481484 , 13.122259 ,
       26.428932 , 36.419918 , 20.250132 , 27.720537 , 20.862013 ,
       10.435287 , 13.758688 , 23.886272 , 22.793003 , 25.457487 ,
       18.77649  , 16.749672 , 27.631205 , 20.438019 , 20.555775 ,
       15.694169 , 22.01195  , 34.631477 , 23.19464  , 19.960718 ,
       36.91764  , 19.84063  , 20.156488 , 23.873333 , 22.944424 ,
       26.093552 , 15.213957 , 25.059044 , 16.493881 , 37.269726 ,
       18.678185 , 20.996134 , 33.840298 , 18.779013 , 13.989851 ,
       28.149147 , 41.191246 , 13.911869 , 10.397522 , 37.930862 ,
       25.388594 , 20.849201 , 19.822563 , 47.18016  , 26.973064 ,
       27.059488 , 18.109674 , 20.904837 , 17.548641 , 17.95324  ,
       15.129455 , 23.28442  , 19.300022 , 28.989714 , 28.642208 ,
       19.269335 , 19.907454 , 15.362327 , 22.326698 , 18.600105 ,
       29.993444 , 43.48818  , 32.133904 , 23.377958 , 20.1670

In [ ]:
reg.score(Xtest, Ytest) # default: R2

0.8944233876354999

In [11]:
y.mean()

np.float64(22.532806324110677)

In [12]:
MSE(Ytest, reg.predict(Xtest))

9.824313950042608

In [13]:
reg.feature_importances_
# score for the importance of different features
# use selectfromModel to do the feature engineering

array([0.03220847, 0.00121258, 0.01346061, 0.00139508, 0.02996123,
       0.3707968 , 0.011302  , 0.1026297 , 0.03054142, 0.0599    ,
       0.0416723 , 0.00973427, 0.29518563], dtype=float32)

### 3. Cross-validation, compare lienar regression with random forest

In [17]:
reg = XGBR(n_estimators=10) # Notice: we import un-training model when doing cross-validation (no .fit())
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [19]:
CVS(reg, Xtrain, Ytrain, cv=5).mean()
# metric: same metric used in reg - default R2

np.float64(0.7591335319367059)

In [20]:
# CV for Random Forest
rfr = RFR(n_estimators=10)
CVS(rfr, Xtrain, Ytrain, cv=5).mean()

np.float64(0.7729594686894425)

In [21]:
CVS(rfr, Xtrain, Ytrain, cv=5, scoring='neg_mean_squared_error').mean()

np.float64(-17.95632847082495)

In [23]:
# parameter: silent - whether output training results
reg = XGBR(n_estimators=10, silent=False)
CVS(reg, Xtrain, Ytrain, cv=5, scoring='neg_root_mean_squared_error').mean()

/Users/c2yao/miniconda3/envs/xgboost/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [10:20:34] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1745056743506/work/src/learner.cc:738: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/c2yao/miniconda3/envs/xgboost/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [10:20:35] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1745056743506/work/src/learner.cc:738: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/c2yao/miniconda3/envs/xgboost/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [10:20:35] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1745056743506/work/src/learner.cc:738: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/c2yao/miniconda3/envs/xgboost/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [10:2

np.float64(-4.337496603294272)

### 4. Plot the Learning Curve

In [24]:
def plot_learning_curve(estimator,title, X, y, 
                        ax=None, #选择子图
                        ylim=None, #设置纵坐标的取值范围
                        cv=None, #交叉验证
                        n_jobs=None #设定索要使用的线程
                       ):
    
    from sklearn.model_selection import learning_curve
    import matplotlib.pyplot as plt
    import numpy as np
    
    train_sizes, train_scores, test_scores = learning_curve(estimator, X, y
                                                            ,shuffle=True
                                                            ,cv=cv
                                                            ,random_state=420
                                                            ,n_jobs=n_jobs)      
    if ax == None:
        ax = plt.gca()
    else:
        ax = plt.figure()
    ax.set_title(title)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_xlabel("Training examples")
    ax.set_ylabel("Score")
    ax.grid() #绘制网格，不是必须
    ax.plot(train_sizes, np.mean(train_scores, axis=1), 'o-'
            , color="r",label="Training score")
    ax.plot(train_sizes, np.mean(test_scores, axis=1), 'o-'
            , color="g",label="Test score")
    ax.legend(loc="best")
    return ax

In [25]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [26]:
plot_learning_curve(XGBR(n_estimators=100, random_state=420),
                    "XGB", Xtrain, Ytrain, ax=None, cv=cv)
plt.show()

KeyboardInterrupt: 

Note: obvious overfitting problem

### 6. Use the Learning Curve to Observe the Influence of n_estimators on the Model Performance

In [ ]:
axisx = range(10, 1050, 50)
rs = []
for i in axisx:
    reg = XGBR(n_estimators=i, random_state=420)
    rs.append(CVS(reg, Xtrain, Ytrain, cv=cv).mean())

print(axisx[rs.index((max(rs))), max(rs)])
plt.fiture(figsize=(20, 5))
plt.plot(axisx, rs, c="red", label="XGB")
plt.legend()
plt.show()

### 7. Progressive Learning Curve: Variance and Generalization Error

$$E(f;D) = bias^2 + var + \epsilon^2$$

In [ ]:
axisx = range(50, 1050, 50)
rs  = [] # store R2 => bias
var = [] # var
ge  = [] # generalized error

for i in axisx:
    reg = XGBR(n_estimators=i, random_state=420)
    cvresult = CVS(reg, Xtrain, Ytrain, cv=cv)

    # record1: bias
    rs.append(cvresult.mean())
    # record2: var
    var.append(cvresult.var())
    # record: ge
    ge.append((1 - cvresult.mean())**2 + cvresult.var())

print(axisx[rs.index(max(rs)), max(rs), var[rs.index(max(rs))]])
print(axisx[var.index(min(var))], rs[var.index(min(var))], min(var))
print(axisx[ge.index(min(ge)), rs[var.index(min(var))], var[rs.index(min(ge))]])